### Build Driver Standings 

In [0]:
SELECT *
FROM formula1.gold.fact_session_results
LIMIT 10;

SELECT *
FROM formula1.gold.dim_drivers
LIMIT 10;

**Requirements**
Creating view that supports Drivers standings and total_points, shows race_starts, number_of_wins, number_of_podium. 
drivers recods should be grouped by season, driver_id, driver_name, driver_nationality, 
Ranked () Partition by season and  order by total_points, number_of_wins

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_driver_standings AS 
WITH dirver_session_summery AS (
  SELECT
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality,
    COUNT(*) AS race_starts,
    SUM(r.points) AS total_points,
    count_if(r.is_win) as number_of_wins,
    count_if(r.is_podium) as number_of_podium
  FROM
    formula1.gold.fact_session_results as r
      JOIN formula1.gold.dim_drivers as d
        ON r.driver_id = d.driver_id
  GROUP BY
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality
)
SELECT
  season,
  driver_id,
  driver_name,
  nationality,
  rank() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS Standing,
  race_starts,
  total_points,
  number_of_wins,
  number_of_podium
FROM
  dirver_session_summery

In [0]:
SELECT driver_name, Standing FROM formula1.gold.v_driver_standings WHERE season = 2021 ORDER BY Standing